# Prepare QA Pairs from Test PDF

Downloads the FAQ PDF from MinIO, extracts numbered Q&A pairs, and saves them as `qa_pairs.json`.
This file is shared across all experiments — run once, then each experiment notebook loads it.

**Output:** `qa_pairs.json` — list of `{"question": "...", "reference": "..."}`

In [1]:
!pip install -q boto3 pdfplumber

In [2]:
import io
import json
import re

import boto3
import pdfplumber

In [3]:
# MinIO configuration (llama-stack-rag namespace)
MINIO_ENDPOINT = "http://minio.llama-stack-rag.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio_rag_user"
MINIO_SECRET_KEY = "minio_rag_password"
EVAL_BUCKET = "eval-data"

## Download PDF from MinIO

In [4]:
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1",
)

objects = s3.list_objects_v2(Bucket=EVAL_BUCKET)
pdf_files = [obj["Key"] for obj in objects.get("Contents", []) if obj["Key"].endswith(".pdf")]
print(f"PDF files in '{EVAL_BUCKET}' bucket: {pdf_files}")

pdf_key = pdf_files[0]
print(f"\nDownloading: {pdf_key}")
pdf_obj = s3.get_object(Bucket=EVAL_BUCKET, Key=pdf_key)
pdf_bytes = pdf_obj["Body"].read()
print(f"Downloaded {len(pdf_bytes)} bytes")

PDF files in 'eval-data' bucket: ['evaldata_najcastejsie-otazky-penazenka-zdravia.pdf']

Downloading: evaldata_najcastejsie-otazky-penazenka-zdravia.pdf
Downloaded 521974 bytes


## Parse PDF and Extract Q&A Pairs

In [5]:
section_headers = set()

with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
    print(f"PDF has {len(pdf.pages)} pages")

    # Pass 1: detect section headers (font size >= 14, non-italic, not a question)
    for page in pdf.pages:
        chars = page.chars
        if not chars:
            continue
        lines_dict = {}
        for c in chars:
            top_key = round(c["top"], 0)
            matched = False
            for existing_top in list(lines_dict.keys()):
                if abs(existing_top - top_key) <= 2:
                    lines_dict[existing_top].append(c)
                    matched = True
                    break
            if not matched:
                lines_dict[top_key] = [c]

        for top_key in sorted(lines_dict.keys()):
            line_chars = sorted(lines_dict[top_key], key=lambda c: c["x0"])
            text = "".join(c["text"] for c in line_chars).strip()
            if not text:
                continue
            max_size = max(c.get("size", 0) for c in line_chars if c["text"].strip())
            is_italic = any(
                "italic" in (c.get("fontname", "") or "").lower()
                for c in line_chars if c["text"].strip()
            )
            if max_size >= 14 and not is_italic and not re.match(r"\d+\.", text) and "?" not in text:
                section_headers.add(text)

    print(f"Detected {len(section_headers)} section header lines to exclude:")
    for h in sorted(section_headers):
        print(f"  - {h}")

    # Pass 2: extract full text, skipping section headers
    full_text = ""
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            cleaned_lines = []
            for line in page_text.split("\n"):
                if line.strip() not in section_headers:
                    cleaned_lines.append(line)
            full_text += "\n".join(cleaned_lines) + "\n"

print(f"\nExtracted {len(full_text)} characters (after removing section headers)")
print(f"\nFirst 500 chars:\n{full_text[:500]}")

PDF has 33 pages
Detected 16 section header lines to exclude:
  - Diabetes
  - Lieky
  - Oči
  - Peňaženka zdravia - Najčastejšie otázky
  - Peňaženka zdravia MAXI
  - Peňaženka zdravia MINI
  - Pohyb medzi MINI a MAXI
  - Prevencia a očkovanie
  - Založenie skupiny
  - Zmeny v skupine
  - Zuby
  - Základné informácie
  - Úhrada finančného príspevku
  - Čerpanie finančného príspevku
  - Čerpanie finančných príspevkov
  - Ženy a deti

Extracted 73275 characters (after removing section headers)

First 500 chars:
1. Kde nájdem Peňaženku zdravia? Je spoplatnená?
Peňaženku zdravia nájdete vo voľne dostupnej mobilnej aplikácii VšZP a v ePobočke. Využívať ju
môžete po zaregistrovaní a aktivácii týchto elektronických služieb.
Stačí si v mobilnej aplikácii alebo ePobočke zvoliť Peňaženku zdravia, v ktorej môžete podávať žiadosti
o finančné príspevky, získať informácie o stave žiadosti, čerpaní alebo aktuálnom zostatku finančného
príspevku za kalendárny rok.
Mobilná aplikácia je dostupná v App S

In [6]:
pattern = r'(\d+)\.\s+(.+?\?)\s*\n(.*?)(?=\n\d+\.\s|\\Z)'
matches = re.findall(pattern, full_text, re.DOTALL)

qa_pairs = []
for num, question, answer in matches:
    question = question.strip()
    answer = answer.strip()
    answer = re.sub(r'\n+', ' ', answer)
    answer = re.sub(r'\s+', ' ', answer)
    if len(answer) > 10:
        qa_pairs.append({"question": question, "reference": answer})

print(f"Extracted {len(qa_pairs)} Q&A pairs\n")
for i, qa in enumerate(qa_pairs[:5]):
    print(f"Q{i+1}: {qa['question']}")
    print(f"A{i+1}: {qa['reference'][:150]}...\n")

Extracted 181 Q&A pairs

Q1: Kde nájdem Peňaženku zdravia? Je spoplatnená?
A1: Peňaženku zdravia nájdete vo voľne dostupnej mobilnej aplikácii VšZP a v ePobočke. Využívať ju môžete po zaregistrovaní a aktivácii týchto elektronick...

Q2: Prečo je Peňaženka zdravia viazaná na mobilnú aplikáciu?
A2: Mobilná aplikácia vám umožní rýchly prístup k Peňaženke zdravia, kdekoľvek sa nachádzate. Pokiaľ vám viac vyhovuje webové rozhranie, stačí si aktivova...

Q3: Na čerpanie príspevkov z Peňaženky zdravia musím mať mobilnú aplikáciu,
alebo mi stačí ePobočka?
A3: Na čerpanie príspevkov z Peňaženky zdravia stačí, ak máte zriadenú ePobočku....

Q4: Ak som sa opäť vrátil do VšZP, môžem pre Peňaženku zdravia využívať svoje
staré konto v ePobočke?
A4: Áno, so svojimi pôvodnými prihlasovacími údajmi sa viete prihlásiť v mobilnej aplikácii VšZP aj ePobočke....

Q5: Som váš dlhoročný poistenec, prečo aj ja nemám nárok na Peňaženku
zdravia?
A5: Peňaženka zdravia je určená pre všetkých našich poistencov...

## Save

In [7]:
output = {
    "source_pdf": pdf_key,
    "num_pairs": len(qa_pairs),
    "qa_pairs": qa_pairs,
}

with open("qa_pairs.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved {len(qa_pairs)} Q&A pairs to qa_pairs.json")
print(f"Source PDF: {pdf_key}")

Saved 181 Q&A pairs to qa_pairs.json
Source PDF: evaldata_najcastejsie-otazky-penazenka-zdravia.pdf
